[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/07_tensorflow_practice/07_tensorflow_practice.ipynb)

# 07. TensorFlow / Keras 라이브러리 실습 (선택)

> 이 커리큘럼(02~06)은 최신 Colab 환경에서 바로 실행되도록 **scikit-learn + PyTorch**로 다시 작성되어
> 있습니다. 하지만 원본 강의([모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/))는
> **TensorFlow** 기반이고, 실무에서도 여전히 TensorFlow/Keras를 쓰는 곳이 많습니다. 이 노트북은
> 커리큘럼 본편(00~06)과 별개로, **TensorFlow/Keras 자체를 손에 익히는 라이브러리 실습**입니다.
> PyTorch로 이미 배운 개념(텐서, 자동미분, 신경망 학습 루프)을 TensorFlow로 다시 짜보면서
> 두 프레임워크의 API 차이를 감 잡는 데 목적이 있습니다.

## 왜 필요한가
- **TensorFlow**: Google이 만든 딥러닝 프레임워크. `tf.Tensor` + `GradientTape`로 자동미분을 하고,
  `tf.keras`라는 고수준 API로 신경망을 몇 줄만에 정의·학습할 수 있습니다.
- PyTorch와 개념은 거의 1:1로 대응됩니다: `torch.Tensor` ↔ `tf.Tensor`, `requires_grad` + `.backward()`
  ↔ `tf.GradientTape()`, `nn.Sequential` ↔ `tf.keras.Sequential`.
- 채용 공고나 레거시 코드베이스에서 TensorFlow/Keras를 마주쳤을 때 당황하지 않는 게 이 노트북의 목표입니다.

## 커리큘럼 안에서의 위치
이 노트북은 00~06 순서에 포함되지 않는 **보너스 실습**입니다. `02_linear_regression`,
`04_neural_networks`를 먼저 끝낸 뒤 보는 걸 권장합니다 (같은 예제를 TensorFlow로 다시 풀어보는 구성이라
PyTorch 버전을 먼저 알아야 비교가 됩니다).

## 실습 0. Colab 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
if IN_COLAB:
    !pip install -q tensorflow

import tensorflow as tf
print("TensorFlow version:", tf.__version__)


## 실습 1. 텐서 기본 — `tf.constant` / `tf.Variable`

TensorFlow 2.x는 PyTorch처럼 즉시 실행(eager execution)됩니다. `tf.constant`는 값이 고정된 텐서,
`tf.Variable`은 학습 중 값이 바뀌는(gradient로 업데이트되는) 텐서입니다 — PyTorch의
`requires_grad=True` 텐서에 대응합니다.

In [ ]:
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[5.0, 6.0], [7.0, 8.0]])

print("elementwise:\n", a * b)
print("행렬곱:\n", a @ b)
print("전치:\n", tf.transpose(a))
print("reshape:\n", tf.reshape(a, [4]))


### 1-1. NumPy ↔ TensorFlow 변환

PyTorch에서 `.numpy()` / `torch.from_numpy()`로 왕복하던 것과 동일하게, TensorFlow도
`.numpy()`와 `tf.constant(np_array)`로 NumPy와 자유롭게 오갈 수 있습니다.

In [ ]:
import numpy as np

np_arr = a.numpy()
print(type(np_arr), np_arr)

back_to_tf = tf.constant(np_arr)
print(type(back_to_tf))


## 실습 2. 자동미분 — `tf.GradientTape`

`00_python_essentials`의 PyTorch 자동미분 예제(`requires_grad=True` + `.backward()`)와 같은 예제를
TensorFlow로 풀어봅니다. TensorFlow는 `tf.GradientTape()` 컨텍스트 안에서 실행된 연산만 미분 대상으로
기록합니다 — PyTorch가 기본적으로 모든 연산을 기록하는 것과 다른 지점입니다.

In [ ]:
W = tf.Variable(0.0)
b = tf.Variable(0.0)

x = tf.constant([1.0, 2.0, 3.0, 4.0])
y = tf.constant([5.0, 7.0, 9.0, 11.0])   # y = 2x + 3

with tf.GradientTape() as tape:
    y_pred = W * x + b
    cost = tf.reduce_mean((y_pred - y) ** 2)

dW, db = tape.gradient(cost, [W, b])
print("dCost/dW:", dW.numpy())
print("dCost/db:", db.numpy())


### 2-1. Gradient Descent 루프 — PyTorch 버전과 비교

PyTorch에서는 `with torch.no_grad():` 블록 안에서 파라미터를 직접 업데이트하고 `.grad.zero_()`로
누적된 gradient를 초기화했습니다. TensorFlow에서는 매 반복마다 `tf.GradientTape()`를 새로 열기 때문에
별도의 `zero_grad`가 필요 없습니다 — 대신 `optimizer.apply_gradients()`로 업데이트를 위임하는 경우가
많습니다.

In [ ]:
W = tf.Variable(0.0)
b = tf.Variable(0.0)
lr = 0.01
optimizer = tf.optimizers.SGD(learning_rate=lr)

for epoch in range(200):
    with tf.GradientTape() as tape:
        y_pred = W * x + b
        cost = tf.reduce_mean((y_pred - y) ** 2)

    dW, db = tape.gradient(cost, [W, b])
    optimizer.apply_gradients(zip([dW, db], [W, b]))

    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  cost={cost.numpy():.4f}  W={W.numpy():.3f}  b={b.numpy():.3f}")

print(f"최종: W={W.numpy():.3f}, b={b.numpy():.3f} (목표: W=2, b=3)")


## 실습 3. `tf.keras.Sequential`로 Linear Regression

앞의 저수준 예제를 `tf.keras`의 고수준 API로 다시 짜면 코드가 훨씬 짧아집니다. `Dense(1)` 레이어
하나가 `W * x + b`와 동일한 연산을 수행하고, `model.compile` + `model.fit`이 학습 루프 전체를
대신합니다. — PyTorch의 `nn.Linear(1, 1)` + 직접 짠 학습 루프에 대응됩니다.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(1,))
])
model.compile(optimizer=tf.optimizers.SGD(learning_rate=0.01), loss="mse")

x_train = tf.constant([[1.0], [2.0], [3.0], [4.0]])
y_train = tf.constant([[5.0], [7.0], [9.0], [11.0]])

history = model.fit(x_train, y_train, epochs=200, verbose=0)
print("최종 loss:", history.history["loss"][-1])
print("학습된 가중치 (W, b):", model.layers[0].get_weights())


## 실습 4. `tf.keras`로 XOR 분류 — `04_neural_networks`와 비교

`04_neural_networks`에서 PyTorch로 풀었던 XOR 문제(선형 모델로는 풀리지 않아 은닉층이 필요했던
예제)를 `tf.keras.Sequential`로 다시 풀어봅니다.

In [ ]:
x_xor = tf.constant([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = tf.constant([[0.0], [1.0], [1.0], [0.0]])

xor_model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
xor_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

xor_model.fit(x_xor, y_xor, epochs=500, verbose=0)
loss, acc = xor_model.evaluate(x_xor, y_xor, verbose=0)
print(f"loss={loss:.4f}  accuracy={acc:.2f}")
print("예측:", xor_model.predict(x_xor, verbose=0).round(2).flatten())


## PyTorch ↔ TensorFlow 대응표

| 개념 | PyTorch | TensorFlow/Keras |
|---|---|---|
| 텐서 생성 | `torch.tensor(...)` | `tf.constant(...)` |
| 학습 가능한 텐서 | `torch.tensor(..., requires_grad=True)` | `tf.Variable(...)` |
| 자동미분 기록 범위 | 기본적으로 모든 연산 | `with tf.GradientTape():` 블록 안만 |
| 미분 계산 | `cost.backward()` → `.grad` | `tape.gradient(cost, [변수들])` |
| gradient 초기화 | `param.grad.zero_()` | 불필요 (매번 새 tape) |
| 신경망 정의 | `nn.Sequential(...)` | `tf.keras.Sequential([...])` |
| 학습 루프 | 직접 작성 (`for epoch in ...`) | `model.fit(...)` (또는 직접 작성) |
| 추론 모드 | `model.eval()` + `torch.no_grad()` | `model.predict(...)` |
| NumPy 변환 | `.numpy()` / `torch.from_numpy()` | `.numpy()` / `tf.constant(np_arr)` |

## TensorFlow 연습 문제
1. 실습 2의 `tf.GradientTape` 예제에서 `y = 3 * x - 1`로 바꾸고, `optimizer.learning_rate`를
   0.1로 올려서 몇 epoch 만에 수렴하는지 비교해보세요.
2. 실습 4의 XOR 모델에서 은닉층 뉴런 수를 8 → 2로 줄이면 학습이 되는지/안 되는지 확인하고,
   `04_neural_networks`에서 다룬 "왜 XOR에 은닉층이 필요한가" 설명과 연결해보세요.
3. `tf.Variable`과 `tf.constant`를 같은 연산에 섞어 써보고, `tape.gradient()`가 `tf.constant`에
   대해서는 `None`을 반환하는 것을 확인해보세요 (기본적으로 `tf.Variable`만 추적 대상입니다).

## PyTorch vs TensorFlow — 차이점과 장단점

API 대응표(위)가 "같은 걸 어떻게 다르게 쓰는가"였다면, 이번엔 왜 두 프레임워크가 이런 차이를 갖게 됐는지,
실무에서 뭘 기준으로 고르는지를 정리합니다.

### 실행 모델의 차이

- **PyTorch — define-by-run (동적 그래프)**: 코드를 한 줄씩 실행하면서 그 자리에서 계산 그래프를 만듭니다.
  `if`/`for` 같은 파이썬 제어문을 그대로 써도 되고, `print()`로 중간 텐서 값을 아무 데서나 찍어볼 수
  있어서 디버깅이 일반 파이썬 코드처럼 직관적입니다.
- **TensorFlow — 원래는 define-and-run (정적 그래프), 지금은 eager가 기본**: TF 1.x는 먼저 전체 계산
  그래프를 정의(`tf.placeholder`, `tf.Session`)한 뒤 실행하는 방식이라 디버깅이 어려웠습니다. TF 2.x부터는
  기본이 eager execution(이 노트북에서 쓴 방식)으로 바뀌어 PyTorch와 체감이 비슷해졌지만, 성능이 중요한
  구간은 `@tf.function`으로 다시 그래프로 컴파일하는 옵션이 남아 있습니다 — 이 지점에서 다시 디버깅이
  까다로워질 수 있습니다.

### 장단점 비교

| 기준 | PyTorch | TensorFlow/Keras |
|---|---|---|
| 학습 곡선 | 파이썬스럽고 직관적, 초심자 진입장벽 낮음 | `tf.keras`는 쉽지만, `tf.function`/저수준 API로 갈수록 개념이 늘어남 |
| 디버깅 | 표준 파이썬 디버거로 바로 추적 가능 | eager 모드는 쉬움, 그래프 모드(`tf.function`)는 어려움 |
| 연구/논문 재현 | 최신 논문 공식 구현체 대부분 PyTorch (Hugging Face 기본값도 PyTorch) | 상대적으로 적음, 다만 Google 계열 연구는 TF/JAX 사용 |
| 배포/서빙 | TorchScript, ONNX, PyTorch Mobile — 최근 빠르게 개선 중 | TensorFlow Serving, TFLite(모바일/엣지), TensorFlow.js(브라우저) — 배포 생태계가 더 오래되고 넓음 |
| 하드웨어 가속 | GPU 우선, `torch.compile`로 컴파일 최적화 지원 | GPU + **TPU** 지원이 특히 강함 (Google Cloud와 궁합) |
| API 안정성 | 메이저 버전 간 큰 변화 적음 | TF 1.x → 2.x에서 API가 크게 바뀐 전례가 있음 |
| 프로덕션 채택 | 스타트업/연구팀에서 빠르게 확산 중 | 레거시 대기업 파이프라인, GCP 기반 인프라에 여전히 많음 |

### 언제 뭘 고르면 좋은가

- **연구·프로토타이핑, 최신 논문 재현, Hugging Face 생태계 활용** → PyTorch가 사실상 표준입니다.
  이 커리큘럼(02~06)이 PyTorch를 기본으로 삼은 것도 같은 이유입니다.
- **모바일/브라우저/엣지 배포, TPU를 적극 활용, 기존 TF 기반 레거시 코드 유지보수** → TensorFlow/Keras가
  여전히 유리한 선택지입니다.
- 실무에서는 "회사/팀이 이미 쓰는 프레임워크"가 결정적인 경우가 많습니다 — 이 노트북의 목적은 어느 한쪽을
  추천하는 게 아니라, 다른 프레임워크로 짜인 코드를 만났을 때 개념을 빠르게 옮겨 읽을 수 있게 하는 것입니다.

## 정리
- **TensorFlow**: `tf.Tensor` + `tf.GradientTape`로 자동미분을 수행하고, `tf.keras`로 신경망을
  고수준으로 정의·학습합니다. PyTorch와 개념은 거의 1:1로 대응되지만 API 스타일(특히 gradient tape의
  명시적 범위 지정)이 다릅니다.
- 이 노트북은 `example-projects/`나 `rag-pipeline-practice/`와는 무관한, PyTorch 대비용 별도
  라이브러리 실습입니다. 커리큘럼 본편은 계속 PyTorch를 기준으로 진행됩니다.

**해설/정답**: [07_tensorflow_practice_solutions.ipynb](07_tensorflow_practice_solutions.ipynb)